In [ ]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
plt.rcParams['figure.facecolor'] = 'white'
import matplotlib.colors as colors
import cmocean.cm as cmo
from glob import glob
%config InlineBackend.print_figure_kwargs = {'bbox_inches': None}

In [ ]:
def prepro(ds):
    return ds.isel(y=slice(800, None))

Load grid and data files

In [ ]:
grid_files = ["/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mask.nc", 
              "/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mesh_hgr.nc",
              "/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mesh_zgr.nc"]

In [ ]:
grid = xr.open_mfdataset(grid_files, parallel=True, preprocess=prepro)

In [ ]:
N2_data_filesREF = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-REF08-Nsquared/clim/" 
                                + "/CREG12.L75-REF08_*.5d_Nsquared.nc"))
N2_data_filesFUT = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-FUT08-Nsquared/clim/" 
                                + "/CREG12.L75-FUT08_*.5d_Nsquared.nc"))

In [ ]:
N2REF = xr.open_mfdataset(N2_data_filesREF, parallel=True, preprocess=prepro)
N2FUT = xr.open_mfdataset(N2_data_filesFUT, parallel=True, preprocess=prepro)

Define dz and Coriolis

In [ ]:
dz = grid.e3t_1d.squeeze()
dz = dz.rename({"z": "deptht"})
dz["deptht"] = N2REF.deptht.values
f = np.abs(grid.ff.squeeze())

Compute the first baroclinic Rossby radius of deformation $\frac{1}{\pi f} \int_{z}^{0}N\,dz$

In [ ]:
RoREF = (1 / (np.pi * f)) * (((N2REF)**0.5).sortby("deptht", ascending=False) * dz.sortby("deptht", ascending=False)).sum("deptht", skipna=True)
RoFUT = (1 / (np.pi * f)) * (((N2FUT)**0.5).sortby("deptht", ascending=False) * dz.sortby("deptht", ascending=False)).sum("deptht", skipna=True)

Compute mean and convert from meter to kilometer

In [ ]:
RoREFplot = RoREF.vobn2.mean("time_counter").squeeze().compute() / 1e3
RoFUTplot = RoFUT.vobn2.mean("time_counter").squeeze().compute() / 1e3

In [ ]:
RoREFplot = RoREFplot.assign_coords({"nav_lon": grid.nav_lon, "nav_lat": grid.nav_lat})
RoFUTplot = RoFUTplot.assign_coords({"nav_lon": grid.nav_lon, "nav_lat": grid.nav_lat})

Define levels and colors for contour plots

In [ ]:
levs = np.arange(0, 4, 0.5)
norm = colors.BoundaryNorm(boundaries=levs, ncolors=256)

levs_diff = np.arange(-0.4, 0.41, 0.1)
norm_diff = colors.BoundaryNorm(boundaries=levs_diff, ncolors=256)

Define mean horizontal grid spacing `dx`

In [ ]:
dx = (((grid.e1t + grid.e2t)/2).squeeze() / 1e3).where(RoREFplot>0)

Plot Fig. S1

In [ ]:
fig = plt.figure(figsize=(8, 4))
gs = fig.add_gridspec(3, 16, height_ratios=[1, 0.001, 0.07])

# define axes
ax1 = fig.add_subplot(gs[0, 0:8])
ax2 = fig.add_subplot(gs[0, 8:16])
axcb1 = fig.add_subplot(gs[2, 2:14])

# plot maps
p1 = ax1.pcolormesh(RoREFplot/dx, cmap=cmo.haline_r, norm=norm)

p2 = ax2.pcolormesh(RoFUTplot/dx, cmap=cmo.haline_r, norm=norm)

# add colorbars
cb1 = plt.colorbar(p1, cax=axcb1, orientation="horizontal", extend="both")
axcb1.set_xticks(np.arange(0, 4, 0.5))
axcb1.set_xlabel(r"")

# add labels and titles
[ax.set_xticks([]) for ax in [ax1, ax2]]
[ax.set_yticks([]) for ax in [ax1, ax2]]
ax1.set_title("REF")
ax2.set_title("FUT")
ax2.text(-100, 1170, "Rossby radius / dx", ha="center", fontweight="bold", fontsize=18)

[ax.text(30, 900, t) for ax, t in zip([ax1, ax2], ["(a)", "(b)"])]

plt.subplots_adjust(top=0.8)

plt.savefig("figures/Figure_S1_Ro_over_dx.png", dpi=600)